In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
df = pd.read_csv("data/train.csv")
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin", "SibSp"])
X = df.drop(columns="Survived")
y = df["Survived"]

X["Embarked"] = X["Embarked"].fillna(X["Embarked"].mode()[0])
# какие колонки категориальные
cat_cols = ["Sex", "Embarked"]
# CatBoost ожидает либо индексы, либо имена — передадим имена в Pool


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool  = Pool(X_test,  y_test,  cat_features=cat_cols)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,                 # печатать каждые 100 итераций
    early_stopping_rounds=50,    # аналог od_wait
)
model.fit(train_pool, eval_set=test_pool, use_best_model=True)

# оценка
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("Test AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("Best iteration:", model.get_best_iteration())

CatBoostError: Invalid type for cat_feature[non-default value idx=562,feature_idx=5]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.